# Mechanistic Interpretability - Induction Circuits

We will explore the concept of induction circuits in neural networks, particularly in transformer architectures. Induction circuits are mechanisms that allow models to recognize and replicate patterns in sequences, enabling them to generalize from learned data.

### Load transformer model and tokenizer

In [ ]:
import functools
import torch
import transformers
from jaxtyping import Float, Int
from transformer_lens import (
    HookedTransformer,
    HookedTransformerConfig,
    ActivationCache,
    FactoredMatrix,
    utils,
)
from typing import List
import einops
from transformer_lens.hook_points import HookPoint
from circuitsvis.attention import attention_patterns
from huggingface_hub import hf_hub_download
import plotly.express as px
import numpy as np

device = torch.device(
    "cpu"
    # "mps" if torch.backends.mps.is_available() else "cpu"
)
torch.set_grad_enabled(False) # Saves computation time

# TransformerLens: Introduction

> - Load and run a `HookedTransformer` model
> - Understand the basic architecture of these models
> - Use the model's tokenizer to convert text to tokens, and vice versa
> - Know how to cache activations, and to access activations from the cache
> - Use `circuitsvis` to visualise attention heads

Use gpt2_small.cfg to find the following, for your GPT-2 Small model:

- Number of layers
- Number of heads per layer
- Maximum context window

In [ ]:
# Load HookedTransformer model
gpt2_small = HookedTransformer.from_pretrained("gpt2-small", dtype=torch.float32).to(device)
gpt2_cfg = gpt2_small.cfg

In [ ]:
model_description_text = """## Loading Models

HookedTransformer comes loaded with >40 open source GPT-style models. You can load any of them in with `HookedTransformer.from_pretrained(MODEL_NAME)`. Each model is loaded into the consistent HookedTransformer architecture, designed to be clean, consistent and interpretability-friendly.

For this demo notebook we'll look at GPT-2 Small, an 80M parameter model. To try the model the model out, let's find the loss on this paragraph!"""

loss = gpt2_small(model_description_text, return_type="loss")
print("Model loss:", loss)

In [ ]:
print(gpt2_small.to_str_tokens("gpt2"))
print(gpt2_small.to_str_tokens(["gpt2", "gpt2"]))
print(gpt2_small.to_tokens("gpt2"))
print(gpt2_small.to_string([50256,    70,   457,    17]))

### Number of correct predictions

In [ ]:
logits = gpt2_small(model_description_text, return_type="logits")
prediction = logits.argmax(dim=-1).squeeze()[:-1]

# [1:] is added to shift the tokens to left so we compare the result with next token not the current one
expected = gpt2_small.to_tokens(model_description_text).squeeze()[1:]

is_correct = prediction == expected
# Count matching values (compare only the overlapping portion)
print(f"Model accuracy: {is_correct.sum()}/{len(expected)}")
print(f"Correct tokens: {gpt2_small.to_str_tokens(prediction[is_correct])}")

### Caching all activations


In [ ]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on task-specific datasets."
gpt2_tokens = gpt2_small.to_tokens(gpt2_text)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_text, remove_batch_dim=True)

print(type(gpt2_logits), type(gpt2_cache))

### Analyzing the cache

In [ ]:
attn_patterns_shorthand = gpt2_cache["pattern", 0]
attn_patterns_full = gpt2_cache["blocks.0.attn.hook_pattern"]

torch.testing.assert_close(attn_patterns_shorthand, attn_patterns_full)

### Verify activations

In [ ]:
layer0_pattern_from_cache = gpt2_cache["pattern", 0]
layer0_k = gpt2_cache["k", 0]
layer0_q = gpt2_cache["q", 0]

query_pos, n_head, d_head = layer0_q.shape

# Attn score
attn_score = einops.einsum(layer0_q, layer0_k, "query_pos n_head d_head, key_pos n_head d_head -> n_head query_pos key_pos")
attn_score = attn_score / d_head**0.5
mask = torch.tril(torch.ones((query_pos, query_pos))).bool()
attn_score = torch.where(mask, attn_score, -1e9)
attn_score = attn_score.softmax(dim=-1)

torch.testing.assert_close(attn_score, layer0_pattern_from_cache)

### Circuit visualization diagram

In [ ]:
for layer in range(gpt2_cfg.n_layers):
    attn_pattern = gpt2_cache["pattern", layer]
    display(
        attention_patterns(
            tokens=gpt2_small.to_str_tokens(gpt2_text),
            attention=attn_pattern,
        )
    )

### Topics to cover:

- Induction circuits explanation
    - Previous token attention head
    - Current token attention head
    - Induction head attention pattern
    - Implementing detectors
    - Induction head analysis
- Logit attribution diagram for induction heads
- Hooks using HookedTransformer
- Causal intervention/Ablation studies on induction heads 

### Interpreting two layer only model

In [ ]:
cfg = HookedTransformerConfig(
    d_model=768,
    d_head=64,
    n_heads=12,
    n_layers=2,
    n_ctx=2048,
    d_vocab=50278,
    attention_dir="causal",
    attn_only=True,  # defaults to False
    tokenizer_name="EleutherAI/gpt-neox-20b",
    seed=398,
    use_attn_result=True,
    normalization_type=None,  # defaults to "LN", i.e. layernorm with weights & biases
    positional_embedding_type="shortformer",
)
REPO_ID = "callummcdougall/attn_only_2L_half"
FILE_NAME = "attn_only_2L_half.pth"
weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILE_NAME)
model = HookedTransformer(cfg)
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
model.load_state_dict(pretrained_weights)

### Visualize and inspect attention patterns

In [ ]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."

logits, cache = model.run_with_cache(text, remove_batch_dim=True)

In [ ]:
for layer in range(cfg.n_layers):
    attn_pattern = cache["pattern", layer]
    display(
        attention_patterns(
            tokens=model.to_str_tokens(text),
            attention=attn_pattern,
        )
    )

### Observations on attention patterns

Potentially:
- Layer 0 Head 3 - Attends to <endoftext> token that appears at the beginning of the sequence.
- Layer 0 Head 7 - Previous token attention head. Attends to the previous occurrence of the current token.
- Layer 1 Head 4 - Induction head that attends to the next token after the previous occurrence of the current token.
- Layer 1 Head 10 - Same as above.

Before we validate the observations, we should build detectors for the induction heads to confirm their functionality.

In [ ]:
def find_prev_token_attn_heads(cache) -> List[str]:
    threshold = 0.5
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer] # n_heads, query_pos, key_pos
        for head in range(cfg.n_heads):
            score = attn_pattern[head].diag(diagonal=-1).mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

def find_curr_token_attn_heads(cache) -> List[str]:
    threshold = 0.3
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer] # n_heads, query_pos, key_pos
        for head in range(cfg.n_heads):
            score = attn_pattern[head].diag(diagonal=0).mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

def find_first_token_attn_head(cache) -> List[str]:
    threshold = 0.5
    result = []
    for layer in range(cfg.n_layers):
        attn_pattern = cache["pattern", layer]
        for head in range(cfg.n_heads):
            score = attn_pattern[head, :, 0].mean()
            if score >= threshold:
                result.append(f"{layer}.{head}")
    
    return result

prev_token_attn_heads = find_prev_token_attn_heads(cache)
curr_token_attn_heads = find_curr_token_attn_heads(cache)
first_token_attn_heads = find_first_token_attn_head(cache)

print(prev_token_attn_heads)
print(curr_token_attn_heads)
print(first_token_attn_heads)

### Induction head analysis

The theory on induction heads can be found [here](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#definition-of-induction-heads)

In order to prove that we have induction heads and induction circuits present, we must show that -
1. For any sequence with repeated tokens, the attention patterns highlights attention on previous occuring copies.
2. The specific occurance of tokens in the sequence doesn't break the pattern.
3. Token loss decreases for copy of tokens which come later in the sequence. 

In [ ]:
torch.manual_seed(0)

# Ensure model is on the correct device
model = model.to(device)

def generate_random_tokens(batch, seq_len) -> torch.Tensor:
    return torch.randint(low=0, high=model.tokenizer.vocab_size, size=(batch, seq_len), device=device)

def generate_repeated_tokens(batch, seq_len) -> torch.Tensor:
    rand_tokens = generate_random_tokens(batch, seq_len)
    prefix = (torch.ones((batch, 1), device=device) * model.tokenizer.bos_token_id).long()

    return torch.concat([prefix, rand_tokens, rand_tokens], dim=-1)

def run_model_with_repeated_tokens(model, batch, seq_len) -> ActivationCache:
    tokens = generate_repeated_tokens(batch, seq_len)
    logits, cache = model.run_with_cache(tokens)

    return tokens, logits, cache

def get_log_probs(logits, tokens) -> torch.Tensor:
    """
    Extract log probabilities of correct next tokens at each position.
    Uses gather to index logprobs[:, :-1] along vocabulary dimension (dim=-1)
    with tokens[:, 1:], effectively computing logprobs[b, s, tokens[b, s+1]]
    """
    logprobs = logits.log_softmax(dim=-1)
    # Ensure tokens are on the same device as logits
    tokens = tokens.to(logprobs.device)
    correct_logprobs = logprobs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)
    return correct_logprobs
    

seq_len = 50
batch_size = 1
rep_tokens, rep_logits, rep_cache = run_model_with_repeated_tokens(model, batch_size, seq_len)
rep_cache.remove_batch_dim()
rep_str = model.to_str_tokens(rep_tokens)
model.reset_hooks()
log_probs = get_log_probs(rep_logits, rep_tokens)

### Plot log probs vs Sequence length

In [ ]:
import pandas as pd

log_probs_data = log_probs.cpu().numpy()[0]
df = pd.DataFrame({
    'seq_len': range(len(log_probs_data)),
    'log_probs': log_probs_data
})

fig = px.line(df, x="seq_len", y="log_probs", title="Log probs vs Sequence length")
fig.show()

### Observation

- Log probability improves after sequence length >= 50. This shows that induction head is copying tokens from sequence < 50.

### Finding induction heads

In [ ]:
for layer in range(model.cfg.n_layers):
    attn_pattern = rep_cache["pattern", layer]
    display(attention_patterns(tokens=rep_str, attention=attn_pattern))

### Observations on attention patterns

- Layer 0 Head 7 consistently attends to the previous occurrence of the current token, confirming its role as a previous token attention head.
- Layer 1 Head 4 and Layer 1 Head 10 both exhibit attention patterns characteristic of induction heads, attending to the next token following the previous occurrence of the current token.
- The pattern of induction heads shows a shifted diagonal exactly by the length of the initial sequence segment (50 tokens in this case).
- These observations align with the theoretical understanding of induction heads and their function within transformer architectures.

### Induction head detector

In [ ]:
def find_induction_heads(cache) -> List[str]:
    threshold = 0.5
    heads = []
    for layer in range(model.cfg.n_layers):
        attn_pattern = cache["pattern", layer]
        seq_len = (attn_pattern.shape[-1] - 1)//2
        for head in range(model.cfg.n_heads):
            # A shift of -1 is needed because 
            # induction head attends to the token
            # just to right side of the previous copy 
            # of current token (which exists at an offset of seq_len)
            score = attn_pattern[head].diag(diagonal=-(seq_len-1)).mean()
            if score >= threshold:
                heads.append(f"{layer}.{head}")
    
    return heads

find_induction_heads(rep_cache)

### Hooks: intervening activations

Idea is to find induction head score using the hooks that intervene when activations are being processed and capture them in a global variable, so we can use that to do data analysis.

In [ ]:
seq_len = 50
batch_size = 10
rep_tokens_10 = generate_repeated_tokens(batch_size, seq_len)

# create a store to keep track of induction head scores
induction_head_score_store = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

def induction_head_score_hook(
    pattern: Float[torch.Tensor, "batch head_index query_pos key_pos"],
    hook: HookPoint,
):
    layer = hook.layer()
    diagonal = torch.diagonal(pattern, offset=-(seq_len-1), dim1=-2, dim2=-1)
    score = einops.reduce(diagonal, "batch head_index pos -> head_index", "mean")
    induction_head_score_store[layer] = score

# Lambda function to run hook only for layers 
# whose name ends with "pattern"
pattern_hooks_name_filter = lambda hook_name: hook_name.endswith("pattern")

model.run_with_hooks(
    rep_tokens_10,
    return_type=None, # we are not calculating the logits to be efficient
    fwd_hooks=[(pattern_hooks_name_filter, induction_head_score_hook)]
)

px.imshow(
    induction_head_score_store, # (layers x heads)
    labels={"x": "Head", "y": "Layer"},
    title="Induction heads score (avgeraged by Batch dim)",
    color_continuous_scale="RdBu_r"
)

## Logit attribution

We want to answer the following question - 

**How much of the model's performance on some particular task is attributable to each component of the model?**

> Key insight: The residual stream is that the output logits are the sum of the contributions of each layer, and thus the sum of the results of each head. This means we can decompose the output logits into a term coming from each head and directly do attribution like this.

Let's say that our model knows that the token Harry is followed by the token Potter, and we want to figure out how it does this. 
The logits on Harry are `residual @ W_U`. But this is a linear map, and the residual stream is the sum of all previous layers 

$$
logits = residual \cdot W_U
$$

where,
$$
residual = embed + attn\_out\_0 + attn\_out\_1
$$

hence,
$$
logits = residual \cdot W_U
\newline
\boxed{logits = (embed \cdot W_U) + (attn\_out\_0 \cdot W_U) + (attn\_out\_1 \cdot W_U)}
\newline
$$

We can be even more specific, and *just* look at the logit of the Potter token - this corresponds to a column of `W_U`, and so a direction in the residual stream - our logit is now a single number that is the sum of `(embed @ potter_U) + (attn_out_0 @ potter_U) + (attn_out_1 @ potter_U)`. Even better, we can decompose each attention layer output into the sum of the result of each head, and use this to get many terms.

We do this by taking into account the individual contributions from each component. Components can be -
1. Attention heads `L0Hx` writing output to residual stream
2. Attention heads `L1Hx` writing output to residual stream
3. Embedding layer directly writing to residual stream


**Note** - When calculating correct output logits, we will get tensors with a dimension (position - 1,), not (position,) - we remove the final element of the output (logits), and the first element of labels (tokens). This is because we're predicting the next token, and we don't know the token after the final token, so we ignore it.

$$
dim(logit\_attributions) = (seq\_len-1)
$$

In [ ]:
def compute_logit_attributions(
    embed: Float[torch.Tensor, "seq d_model"],
    layer0_output: Float[torch.Tensor, "seq n_heads d_model"],
    layer1_output: Float[torch.Tensor, "seq n_heads d_model"],
    W_U: Float[torch.Tensor, "d_model d_vocab"],
    tokens: Int[torch.Tensor, "seq"],
) -> Float[torch.Tensor, "seq-1 n_components"]:
    # Remove the last token to align with next-token prediction
    embed = embed[:-1]                          # shape: [seq-1, d_model]
    layer0_output = layer0_output[:-1]          # shape: [seq-1, n_heads, d_model]
    layer1_output = layer1_output[:-1]          # shape: [seq-1, n_heads, d_model]
    # Shift tokens by one to align with next-token prediction
    tokens = tokens[1:]                         # shape: [seq-1]
    W_U_subset = W_U[:, tokens].squeeze()       # shape: [d_model, seq-1]

    return torch.concat(
        [
            einops.einsum(embed, W_U_subset, "seq d_model,d_model seq -> seq").unsqueeze(dim=-1),
            einops.einsum(layer0_output, W_U_subset, "seq n_heads d_model,d_model seq -> seq n_heads"),
            einops.einsum(layer1_output, W_U_subset, "seq n_heads d_model,d_model seq -> seq n_heads"),
        ],
        dim=-1
    )

### Visualising the contributions from each component -

In [ ]:
def convert_tokens_to_string(model, tokens, batch_index=0):
    """
    Helper function to convert tokens into a list of strings, for printing.
    """
    if len(tokens.shape) == 2:
        tokens = tokens[batch_index]
    return [f"|{model.tokenizer.decode(tok)}|_{c}" for (c, tok) in enumerate(tokens)]

def plot_logit_attributions(model, logit_attr, tokens):
    """
    Plots logit attributions using plotly.
    """
    x_labels = ["Embed"] + [f"L{l}H{h}" for l in range(model.cfg.n_layers) for h in range(model.cfg.n_heads)]
    y_labels = convert_tokens_to_string(model, tokens[:,:-1])
    title = "Logit Attributions from Different Components"

    img = logit_attr.cpu().numpy()
    fig = px.imshow(
        img,
        y=y_labels,
        x=x_labels,
        labels={"y": "Sequence Position", "x": "Term", "color": "Logit"},
        title=title,
        color_continuous_scale="RdBu_r",
        color_continuous_midpoint=0.0,
        text_auto=".2f",
        width=24*len(x_labels),
        height=100 + (30 if title else 0) + 15 * len(y_labels),
        
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

In [ ]:
text = "We think that powerful, significantly superhuman machine intelligence is more likely than not to be created this century. If current machine learning techniques were scaled up to this level, we think they would by default produce systems that are deceptive or manipulative, and that no solid plans are known for how to avoid this."
logits, cache = model.run_with_cache(text, remove_batch_dim=True)
str_tokens = model.to_str_tokens(text)
tokens = model.to_tokens(text)

with torch.inference_mode():
    embed = cache["embed"]
    layer0_output = cache["result", 0]
    layer1_output = cache["result", 1]
    logit_attr = compute_logit_attributions(embed, layer0_output, layer1_output, model.W_U, tokens.squeeze())

plot_logit_attributions(model, logit_attr, tokens)

In [ ]:
for index, item in enumerate(zip(logit_attr.cpu().numpy()[:,0], str_tokens[:-1])):
    curr_logit, curr_label = item
    if curr_logit > 3.5 and index+1 < len(str_tokens):
        next_label = str_tokens[index+1]
        print(f"{curr_label} -> {next_label}")

### Observations -
- The embedding layer has a significant positive contribution towards predicting the correct next token. Suggesting that bigram frequencies are very important.
- Certain heads in Layer 0 and Layer 1 show strong positive contributions, indicating their importance in the induction mechanism.

Most contributions are coming from very common bigram tokens which can directly be supplied from embedding layer - 

```txt
 super -> human
 more ->  likely
 machine ->  learning
 by ->  default
 manip -> ulative
 to ->  avoid
```

Now, let's look at some contributions from induction heads -


In [ ]:
embed = rep_cache["embed"]
layer0_output = rep_cache["result", 0]
layer1_output = rep_cache["result", 1]
rep_logit_attr = compute_logit_attributions(embed, layer0_output, layer1_output, model.W_U, rep_tokens.squeeze())
plot_logit_attributions(model, rep_logit_attr, rep_tokens)

### Observations -

The first half the graph is mostly meaningless as the sequence is random tokens. The induction heads start contributing meaningfully after the first occurrence of the sequence, which is around token index 50 when the sequence starts repeating.

The contributions come from L1H4 and L1H10, which we identified as induction heads earlier based on their attention patterns. This confirms that these heads are indeed functioning as induction heads, contributing significantly to the model's ability to predict the next token in the repeated sequence.

## Ablation - induction head ablation

This is a causal intervention method that helps confirm if the induction head is indeed responsible for generating the desired result or not.

The idea is to use hooks to intervene the attention head output activations before they are passed into residual stream and do either of the following - 
1. Either, attention_head_output = **0**
2. Or, attention_head_output = **mean**

In [ ]:
def zero_ablation_hook(
    z: Float[torch.Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index: int,
):
    z[:,:,head_index,:] = 0.

def get_ablation_scores(
    model: HookedTransformer,
    tokens: Int[torch.Tensor, "batch seq"],
    ablation_fn = zero_ablation_hook,
):
    # To capture ablation scores
    ablation_scores = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)

    model.reset_hooks()
    seq = (tokens.shape[-1] - 1) // 2
    logits = model(tokens, return_type="logits")
    loss_without_ablation = -get_log_probs(logits, tokens)[:, -(seq-1):].mean()

    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            ablated_logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(utils.get_act_name("z", layer), functools.partial(ablation_fn, head_index=head))]
            )

            loss_with_ablation = -get_log_probs(ablated_logits, tokens)[:, -(seq-1):].mean()

            ablation_scores[layer, head] = loss_with_ablation - loss_without_ablation
    
    return ablation_scores

ablation_scores = get_ablation_scores(model, rep_tokens)

px.imshow(
    ablation_scores,
    x=[f"H{h}" for h in range(model.cfg.n_heads)],
    y=[f"L{l}" for l in range(model.cfg.n_layers)],
    title="Zero ablation scores by layer (=Logloss difference)",
    labels={"x": "Head", "y": "Layer"},
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0.0,
    text_auto=".2f",
)

### Observations:

This graph tells us the importance of each attention head and how much loss differs from the baseline when that head is ablated.

We can see that when L0H7 head is ablated we get the highest difference in the loss. Signifying the highest importance of that attention head. This agrees well with our previous observations where we concluded that L0H7 is the previous token attention head. Which is crucial for the induction circuit to work.

Next, we see that L1H4 and L1H10 also have high loss difference which also agree with our previous observations where we found these to be working as induction heads.

Additionally we observe L0H4 and L0H11 showing up as well, which was not obvious from our previous observations. 

### Mean ablation hook

A more cleaner implementation is to use mean_ablation_hook instead

In [ ]:
def mean_ablation_hook(
    z: Float[torch.Tensor, "batch seq n_heads d_head"],
    hook: HookPoint,
    head_index: int,
):
    z[:,:,head_index,:] = z[:,:,head_index,:].mean(0) # Mean over batch dimension

batch_size = 10
seq_len = 50
rep_tokens_batch = run_model_with_repeated_tokens(model, batch_size, seq_len)[0]
ablation_scores = get_ablation_scores(model, rep_tokens_batch, mean_ablation_hook)
px.imshow(
    ablation_scores,
    x=[f"H{h}" for h in range(model.cfg.n_heads)],
    y=[f"L{l}" for l in range(model.cfg.n_layers)],
    title="Mean ablation scores by layer (=Logloss difference)",
    labels={"x": "Head", "y": "Layer"},
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0.0,
    text_auto=".2f",
)

# Reverse engineering induction circuits

Previous approaches of analysing attention heads using graphs is great for feature analysis but is not a rigorous approach.
We should dig into the weights of the matrices and understand why the attends do what they do.

### OV Copying Circuit - 

We aim to study the `L1H4` and `L1H10` attention heads which we claim to be copying the token at position supplied by `L0H7` attention head.

$$
W_{OV} = W_V \cdot W_O
$$

$W_{OV}^{h}$ has size $(d_\text{model}, d_\text{model})$, it is a linear map describing **what information gets moved from source to destination, in the residual stream.**

In other words, if $x$ is a vector in the residual stream, then $x^T W_{OV}^{h}$ is the vector written to the residual stream at the destination position, if the destination token only pays attention to the source token at the position of the vector $x$.

$$
\boxed{OV\_circuit = W_E \cdot W_V^{(h)} \cdot W_O^{(h)} \cdot W_U}
$$

$W_E W_{OV}^h W_U$ has size $(d_\text{vocab}, d_\text{vocab})$, it is a linear map describing **what information gets moved from source to destination, in a start-to-end sense.**

If $A$ is the one-hot encoding for token `A`, then:

* $A^T W_E$ is the embedding vector for `A`.
* $A^T W_E W_{OV}^h$ is the vector which would get written to the residual stream at the destination position, if the destination token only pays attention to `A`.
* $A^T W_E W_{OV}^h W_U$ is the unembedding of this vector, i.e. the thing which gets added to the final logits.

Because we are studying the attention head with copying behaviour, we expect that the $OV\_circuit$ should approximate an identity mapping for the induction heads. This means that when a token is processed through this circuit, it should ideally reproduce the same token in the output, facilitating the copying mechanism essential for induction.

This means that we should see high values along the diagonal of the OV_circuit matrix, indicating that each token is being mapped to itself.

In [ ]:
head_index = 4
layer_index = 1

W_OV = FactoredMatrix(model.W_V[layer_index, head_index], model.W_O[layer_index, head_index])

full_ov_circuit = model.W_E @ W_OV @ model.W_U
indices = torch.randint(0, model.cfg.d_model, (200,), device=device)
partial_ov_circuit = full_ov_circuit[indices][:, indices].AB

px.imshow(
    partial_ov_circuit.cpu().numpy(),
    title=f"Partial W_E @ W_V{layer_index}.{head_index} @ W_U Circuit",
    labels={"x": "d_model indices", "y": "d_model indices"},
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0.0,
    text_auto=".2f",
    width=600,
    height=600,
)

### Identity mapping summary statistic

We can use accuracy to quantify how closely the OV_circuit approximates an identity mapping. This can be done by checking how many tokens are correctly mapped to themselves when passed through the OV_circuit.

In [ ]:
def top_1_acc(full_ov_circuit: FactoredMatrix):
    total = 0
    
    for indices in torch.split(torch.arange(full_ov_circuit.shape[0], device=device), batch_size):
        AB_slice = full_ov_circuit[indices].AB
        total += (torch.argmax(AB_slice, dim=1) == indices).float().sum().item()

    return total / full_ov_circuit.shape[0]
    

accuracy = top_1_acc(full_ov_circuit)
print(f"Accuracy: {accuracy}")

The accuracy is low (~30%) because we are missing contributions from L1H10 attention head. Let's add those contributions as well and see what happens.

In [ ]:
head_indices = [4, 10]
layer_index = 1

W_O = einops.rearrange(model.W_O[layer_index, head_indices], "head d_head d_model -> (head d_head) d_model")
W_V = einops.rearrange(model.W_V[layer_index, head_indices], "head d_model d_head -> d_model (head d_head)")

W_OV = FactoredMatrix(W_V, W_O)

full_ov_circuit = model.W_E @ W_OV @ model.W_U
accuracy = top_1_acc(full_ov_circuit)
print(f"Accuracy: {accuracy}")

The accuracy is now close to 95% which is much better

### QK Circuit -

Next we study `L0H7` attention head which we claim to be working as a previous token attention head.

$$
\boxed{\large{QK\_circuit = W_\text{pos} \cdot W_Q^{h} \cdot (W_K^{h})^T \cdot W_\text{pos}^T}}
$$

$W_{pos} W_{QK}^h W_{pos}^T$ has size $(n_\text{ctx}, n_\text{ctx})$, it is a bilinear form describing **where information is moved to and from**, among tokens in our context (i.e. which token positions pay attention to other positions).

If $i$ and $j$ are one-hot encodings for positions `i` and `j` (in other words they are just the ith and jth basis vectors), then $i^T W_{pos} W_{QK}^h W_{pos}^T j$ is the attention score paid by the token with position `i` to the token with position `j`:

$$
\large{i^T \, W_{pos}\, W_{QK}^{h}\, W_{pos}^T \, j = \underbrace{(i^T W_{pos} W_Q^{h})}_{\text{query for i-th token}}  \underbrace{(j^T W_{pos} W_K^{h})^T}_{\text{key for j-th token}}}
$$

This means that for `L0H7` to work as a previous token attention head, we should see the following relationship:
$$
pos_i = pos_j - 1
$$

In [ ]:
layer = 0
head_index = 7

W_Q = model.W_Q[layer, head_index]
W_K = model.W_K[layer, head_index]

W_QK = FactoredMatrix(W_Q, W_K.T)
pos_by_pos_pattern = model.W_pos @ W_QK @ model.W_pos.T

mask = torch.tril(torch.ones_like(pos_by_pos_pattern.AB)).bool()
pos_by_pos_pattern = torch.where(mask, pos_by_pos_pattern.AB / model.cfg.d_head ** 0.5, -1e6).softmax(-1)

px.imshow(
    pos_by_pos_pattern.cpu().numpy()[:200, :200],
    labels={"x": "Key", "y":"Query"},
    title="Attention patterns for previous token QK circuit",
    width=700,
    height=600,
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu_r"
)

### Observation:
The graph shows -1 diagonal. Which means that indeed the attention head is paying attention to previous token from the current token position. 

## K-composition

Similar to what we did for logit attributions (i.e studying individual contribution from each component in previous layers) we can do the same here.

$$
\large{x \cdot W_Q^{1.H} = (embed + pos\_embed + \sum_{h=0}^{11}{x^{0.h}}) \cdot W_Q^{1.H}}
\newline
\newline
\boxed{\large{x \cdot W_Q^{1.H} = {\sum_{i=0}^{13}{y_i}} \cdot W_Q^{1.H}}}
$$

with each $y_i$ having shape `[seq, d_model]`, and the sum of $y_i$s being the full residual stream `x`. 

<img src="./images/q-composition.png" width="600">

There are total 14 terms and we should now study the relative importance of each term. One way to study is to use `norms` as the metric. This is a very high level metric and can be be dodgy but it will give us some information.

In [ ]:
def decompose_qk_input(cache: ActivationCache) -> Float[torch.Tensor, "n_heads+2 seq d_model"]:
    embed = cache["embed"].unsqueeze(0)              # shape: [1, seq, d_model]
    pos_embed = cache["pos_embed"].unsqueeze(0)      # shape: [1, seq, d_model]
    result = cache["result", 0].transpose(0, 1)      # shape: [n_heads, seq, d_model]
    
    return torch.concat([embed, pos_embed, result], dim=0)

def decompose_q(
    decomposed_qk_input: Float[torch.Tensor, "n_heads+2 seq d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[torch.Tensor, "n seq d_head"]:
    return einops.einsum(
        decomposed_qk_input, 
        model.W_Q[1, ind_head_index], 
        "n seq d_model,d_model d_head -> n seq d_head"
    )

def decompose_k(
    decomposed_qk_input: Float[torch.Tensor, "n_heads+2 seq d_model"],
    ind_head_index: int,
    model: HookedTransformer,
) -> Float[torch.Tensor, "n seq d_head"]:
    return einops.einsum(
        decomposed_qk_input, 
        model.W_K[1, ind_head_index], 
        "n seq d_model,d_model d_head -> n seq d_head"
    )

### Get decomposed tensors for analysis

In [ ]:
# Recompute rep tokens/logits/cache, if we haven't already
seq_len = 50
batch_size = 1
rep_tokens, rep_logits, rep_cache = run_model_with_repeated_tokens(
    model, 
    batch_size,
    seq_len, 
)
rep_cache.remove_batch_dim()

ind_head_index = 4 # can be changed to 10, if required.

# First we get decomposed q and k input, and check they're what we expect
decomposed_qk_input = decompose_qk_input(rep_cache)
decomposed_q = decompose_q(decomposed_qk_input, ind_head_index, model)
decomposed_k = decompose_k(decomposed_qk_input, ind_head_index, model)

# Verify the results match
torch.testing.assert_close(
    decomposed_qk_input.sum(0),
    rep_cache["resid_pre", 1] + rep_cache["pos_embed"],
    rtol=0.01,
    atol=1e-05,
)
torch.testing.assert_close(
    decomposed_q.sum(0), rep_cache["q", 1][:, ind_head_index], rtol=0.01, atol=0.001
)
torch.testing.assert_close(
    decomposed_k.sum(0), rep_cache["k", 1][:, ind_head_index], rtol=0.01, atol=0.01
)

### Plot Q & K decompostion results

In [ ]:
component_labels = ["Embed", "PosEmbed"] + [f"L0H{head}" for head in range(model.cfg.n_heads)]
for decomposed_input, name in [(decomposed_q, "query"), (decomposed_k, "key")]:
    fig = px.imshow(
        decomposed_input.pow(2).sum([-1]).cpu().numpy(),
        labels={"x": "Position", "y": "Component"},
        title=f"Norms of components of {name}",
        y=component_labels,
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu_r",
    )
    fig.show()

Clearly we see that `Embed`, `PosEmbed` and `L0H7` playing a crucial role in `L1` induction heads.

Next, lets try to study the attention scores from each of the components from previous layer.

### Decompose attention scores

We shall consider attention scores and not attention pattern for analysis.

**Note** -

$$
Attention\_score = \frac{QK^T}{\sqrt{d_{head}}}, \space Attention\_pattern = softmax(\frac{QK^T}{\sqrt{d_{head}}})
$$

Because $Attention\_pattern$ is non-linear, we can't really use that directly assume that it can linearly decompose. Hence we chose $Attention\_score$ instead.

In [ ]:
def decompose_attn_scores(
    decomposed_q: Float[torch.Tensor, "q_comp q_pos d_head"],
    decomposed_k: Float[torch.Tensor, "k_comp k_pos d_head"],
    model: HookedTransformer,
) -> Float[torch.Tensor, "q_comp k_comp query_pos key_pos"]:
    return einops.einsum(
        decomposed_q, 
        decomposed_k, 
        "q_comp q_pos d_head, k_comp k_pos d_head -> q_comp k_comp q_pos k_pos"
    ) / model.cfg.d_head ** 0.5


decomposed_attn_scores = decompose_attn_scores(decomposed_q, decomposed_k, model)

### Attention score decomposition: Query vs Key pos 

In [ ]:
q_label, k_label = "Embed", "L0H7"
decomposed_attn_scores_pair = decomposed_attn_scores[component_labels.index(q_label), component_labels.index(k_label)]
decomposed_attn_scores_pair = torch.tril(decomposed_attn_scores_pair)
px.imshow(
    decomposed_attn_scores_pair,
    title=f"Attn score contribution from query={q_label} and key={k_label}",
    labels={"x": "Key Position", "y": "Query Position"},
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu_r",
)

### Attention score decomposition: Std deviation of scores

In [ ]:
decomposed_stds = decomposed_attn_scores.std(dim=[-2, -1])
px.imshow(
    decomposed_stds,
    x=component_labels,
    y=component_labels,
    labels={"x": "Query component", "y": "Key component"},
    color_continuous_midpoint=0.0,
    color_continuous_scale="RdBu_r",
    width=700
)

### Observations:

1. `L0H7` which is the previous token attention head, uses `Embedding` layer
2. `L1H4` and `L1H10` attention heads are induction heads that rely on `L0H7` head to figure out which token position to copy information from.

## Interpretting the full circuit

Ignoring all other noisy components, we can assume that the complete circuit is as follows -

$$
full\_K\_comp\_circuit = W_{E} \cdot W_{QK}^{1.4} \cdot (W_{OV}^{0.7})^T \cdot W_{E}^T
$$

We should expect this circuit to be an identity mapping circuit, i.e high diagonal values.

In [ ]:
def compute_full_k_comp_circuit(
    model: HookedTransformer,
    prev_token_head_index: int,
    ind_head_index: int,
) -> Float[torch.Tensor, "d_vocab d_vocab"]:
    W_E = model.W_E
    W_QK = FactoredMatrix(model.W_Q[1, ind_head_index], model.W_K[1, ind_head_index].T)
    W_OV = FactoredMatrix(model.W_V[0, prev_token_head_index], model.W_O[0, prev_token_head_index])

    return W_E @ (W_OV @ W_QK.T) @ W_E.T

full_circuit = compute_full_k_comp_circuit(model, prev_token_head_index=7, ind_head_index=4)
accuracy = top_1_acc(full_circuit)
print(f"Accuracy: {accuracy}")

### Composition scores:

Explainer diagram:
<details>
<img src="./images/compositional_scores.png">
</details>

The key idea of compositional scores is that the residual stream is a large space, and each head is reading and writing from small subspaces. By default, any two heads will have little overlap between their subspaces (in the same way that any two random vectors have almost zero dot product in a large vector space). But if two heads are deliberately composing, then they will likely want to ensure they write and read from similar subspaces, so that minimal information is lost. As a result, we can just directly look at "how much overlap there is" between the output space of the earlier head and the K, Q, or V input space of the later head.

This leads to a phenomenon called [Virtual weights](https://transformer-circuits.pub/2021/framework/index.html#residual-comms) 
Example - $W_O^2 \cdot W_I^3$.

A formal way to determine the extent of such interactions is -

$$
\boxed{comp\_score = \frac{\parallel W_A \cdot W_B \parallel_F}{\parallel W_A \parallel_F \cdot \parallel W_B \parallel_F}}
$$

Where, $\parallel . \parallel_F$ is the [Frobenius norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.matrix_norm.html#torch.linalg.matrix_norm) of a matrix.

Why is this a good metric ?
Theorem in Linear algebra theorem states that Frobenius norm of a matrix is equal to the sum of square of singular values of that matrix.
Below is the proof.
<details>
<summary>Proof</summary>

Frobenius norm can be defined as follows:
$$
\|A\|_{F}^{2}=\sum _{i,j}|a_{ij}|^{2}=\text{tr}(A^{T}A)
$$

SVD of $A$ matrix is defined as follows:
$$
A=U\Sigma V^{T}

\begin{aligned}

\|A\|_F^2 &= \text{tr}(A^T A) \ &= \text{tr}((U\Sigma V^T)^T (U\Sigma V^T)) \ &= \text{tr}(V\Sigma^T U^T U\Sigma V^T)

\end{aligned}
$$

Since $U$ is an orthogonal matrix, and $U^{T}U=I$, then
$$
\begin{aligned} \|A\|_F^2 &= \text{tr}(V\Sigma^T I \Sigma V^T) \ &= \text{tr}(V\Sigma^T \Sigma V^T) \end{aligned}
$$

Using the cyclic property of the trace, $text{tr}(ABC)=\text{tr}(BCA)$
$$
\begin{aligned}\|A\|_{F}^{2}&=\text{tr}(\Sigma ^{T}\Sigma V^{T}V)\end{aligned}
$$

Since $V$ is an orthogonal matrix, $V^{T}V=I$:
$$
\begin{aligned}\|A\|_{F}^{2}&=\text{tr}(\Sigma ^{T}\Sigma )\end{aligned}
$$

Because $\Sigma$ is a diagonal matrix with entries $\sigma _{1},\sigma _{2},\dots ,\sigma _{k}$, the product $\Sigma ^{T}\Sigma$ is a diagonal matrix with entries $\sigma _{1}^{2},\sigma _{2}^{2},\dots ,\sigma _{k}^{2}$. The trace is the sum of these diagonal entries:
$$
\boxed{\|A\|_{F}^{2}=\sum _{i=1}^{\min (m,n)}\sigma _{i}^{2}}
$$

</details>

Thus, 
$$
W_A=U_AS_AV_A^T
\newline
W_B=U_BS_BV_B^T
\newline
\parallel W_A \parallel_F=\parallel S_A \parallel_F
\newline
\parallel W_B \parallel_F=\parallel S_B \parallel_F
\newline
\boxed{\parallel W_A W_B \parallel_F=\parallel S_A V_A^T U_B S_B \parallel_F}
$$ 

In some sense, 
$$
V_A^T U_B
$$ 

represents how aligned the subspaces written to and read from are, and the $S_A$ and $S_B$ terms weights by the importance of those subspaces.

In [ ]:
def compute_compositional_score(W_A: Float[torch.Tensor, "in_A out_A"], W_B: Float[torch.Tensor, "out_A out_B"]) -> float:
    W_A_F = W_A.pow(2).sum().sqrt()
    W_B_F = W_B.pow(2).sum().sqrt()
    W_AB_F = (W_A @ W_B).pow(2).sum().sqrt()

    return (W_AB_F / (W_A_F * W_B_F)).item()

def generate_single_random_comp_score() -> float:
    """
    Write a function which generates a single composition score for random matrices
    """
    W_A_left = torch.empty(model.cfg.d_model, model.cfg.d_head)
    W_B_left = torch.empty(model.cfg.d_model, model.cfg.d_head)
    W_A_right = torch.empty(model.cfg.d_model, model.cfg.d_head)
    W_B_right = torch.empty(model.cfg.d_model, model.cfg.d_head)

    for W in [W_A_left, W_B_left, W_A_right, W_B_right]:
        torch.nn.init.kaiming_uniform_(W, a=np.sqrt(5))

    W_A = W_A_left @ W_A_right.T
    W_B = W_B_left @ W_B_right.T

    return compute_compositional_score(W_A, W_B)

n_samples = 300
comp_scores_baseline = np.zeros(n_samples)
for i in range(n_samples):
    comp_scores_baseline[i] = generate_single_random_comp_score()

print("\nMean:", comp_scores_baseline.mean())
print("Std:", comp_scores_baseline.std())

W_QK = model.W_Q @ model.W_K.transpose(-1, -2)
W_OV = model.W_V @ model.W_O

composition_scores = {
    "Q": torch.zeros((model.cfg.n_heads, model.cfg.n_heads)),
    "K": torch.zeros((model.cfg.n_heads, model.cfg.n_heads)),
    "V": torch.zeros((model.cfg.n_heads, model.cfg.n_heads)),
}

for i in range(model.cfg.n_heads):
    for j in range(model.cfg.n_heads):
        composition_scores["Q"][i, j] = compute_compositional_score(W_QK[1, j], W_OV[0, i])
        composition_scores["K"][i, j] = compute_compositional_score(W_QK[1, j].T, W_OV[0, i])
        composition_scores["V"][i, j] = compute_compositional_score(W_OV[1, j], W_OV[0, i])

for comp_type in ["Q", "K", "V"]:
    fig = px.imshow(
        composition_scores[comp_type],
        color_continuous_midpoint=comp_scores_baseline.mean(),
        color_continuous_scale="RdBu_r",
        labels={"x": "Head", "y": "Head"},
        title=f"{comp_type}-composition scores for layer 1",
        x=[f"H{h}" for h in range(model.cfg.n_heads)],
        y=[f"H{h}" for h in range(model.cfg.n_heads)],
    )
    fig.show()